### Q.1 Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [34]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

def merge_fields(example):
    example["combined_text"] = f"{example['prompt']} {example['A']}"
    return example

dataset = dataset.map(merge_fields)

print(dataset)
print(dataset.column_names)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text'],
    num_rows: 2000
})
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text']


### Q.2 Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer? 

In [35]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

vocab_count = bert_tokenizer.vocab_size

print(f"Vocabulary Size = {vocab_count}")

Vocabulary Size = 30522


### Q.3 Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token. 

In [36]:
sep_identifier = bert_tokenizer.convert_tokens_to_ids("[SEP]")

print(f"[SEP] Token ID : {sep_identifier}")

[SEP] Token ID : 102


### Q.4 Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

### What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [38]:
import torch

# Convert every prompt to a clean string
all_prompts = [str(x) if x is not None else "" for x in dataset["prompt"]]

encoded_batch = bert_tokenizer(
    all_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

attention_mask = encoded_batch["attention_mask"]

token_lengths = attention_mask.sum(dim=1)

avg_length = token_lengths.float().mean()

print(f"Average token length : {avg_length.item():.2f}")

Average token length : 25.93


### Q.5 A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

### In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [ ]:
hidden_dimension = 768
head_count = 12

dimension_per_head = hidden_dimension // head_count

print(f"Dimensions handled by each attention head : {dimension_per_head}")

### Q.6 Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

### What is the exact shape of the last_hidden_state tensor returned? 

* Note: We follow zero-indexing here.

In [ ]:
from transformers import AutoModel

encoder = AutoModel.from_pretrained("bert-base-uncased")

sentence = dataset[0]["prompt"]

tokenized = bert_tokenizer(
    sentence,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = encoder(**tokenized)

hidden_tensor = outputs.last_hidden_state

print("Tensor Shape :", hidden_tensor.shape)

### Q.7 Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [ ]:
cls_vector = model_output.last_hidden_state.squeeze(0)[0]

partial_sum = cls_vector.narrow(0, 0, 5).sum().item()

print(
    f"Sum of the first five elements of the CLS embedding = {partial_sum:.4f}"
)

### Q.8 Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

### What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [ ]:
from transformers import AutoTokenizer, AutoModel

text = "Light-ion fusion is a technique."

attention_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

attention_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

encoded = attention_tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    result = attention_model(**encoded)

last_layer = result.attentions[-1][0, 0]

tokens = attention_tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

print("Tokens\n")
for token in tokens:
    print(token)

print("\nAttention Matrix Shape:", tuple(last_layer.shape))

### Q.9 Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

question = train_df.at[0, "prompt"]
choice_b = train_df.at[0, "B"]

embeddings = embedder.encode(
    [question, choice_b],
    convert_to_numpy=True
)

score = cosine_similarity(
    embeddings[:1],
    embeddings[1:]
)[0][0]

print(
    f"Cosine Similarity = {score:.4f}"
)

### Q.10 Build two complete ranking pipelines evaluating every row in train.csv.

* Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

* Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

* First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

### Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

choices = ["A", "B", "C", "D", "E"]

train_df["answer_text"] = train_df.apply(
    lambda r: r[r["answer"]],
    axis=1
)

###################################################
# TF-IDF
###################################################

vectorizer = TfidfVectorizer()

documents = train_df["prompt"].tolist()

for c in choices:
    documents.extend(train_df[c].tolist())

vectorizer.fit(documents)

tfidf_correct = 0

for row in train_df.itertuples():

    q_vec = vectorizer.transform([row.prompt])

    sims = []

    for c in choices:
        option = getattr(row, c)

        sim = cosine_similarity(
            q_vec,
            vectorizer.transform([option])
        )[0][0]

        sims.append(sim)

    pred = choices[np.argmax(sims)]

    if pred == row.answer:
        tfidf_correct += 1

###################################################
# Sentence Transformer
###################################################

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert_correct = 0

for row in train_df.itertuples():

    texts = [row.prompt] + [getattr(row, c) for c in choices]

    vectors = encoder.encode(
        texts,
        convert_to_numpy=True
    )

    q = vectors[0].reshape(1, -1)

    similarities = cosine_similarity(
        q,
        vectors[1:]
    )[0]

    pred = choices[np.argmax(similarities)]

    if pred == row.answer:
        sbert_correct += 1

print(
    f"TF-IDF Accuracy : {tfidf_correct/len(train_df):.4f}"
)

print(
    f"SBERT Accuracy  : {sbert_correct/len(train_df):.4f}"
)

### Q.11 Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [ ]:
from transformers import pipeline

zs_classifier = pipeline(
    task="zero-shot-classification"
)

sample = train_df.iloc[1]

labels = [
    sample["A"],
    sample["B"],
    sample["C"]
]

prediction = zs_classifier(
    sample["prompt"],
    labels
)

best_probability = max(prediction["scores"])

print(
    f"Highest Score = {best_probability:.4f}"
)

### Q.12 Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

### What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [ ]:
multi_result = zs_classifier(
    sample["prompt"],
    labels,
    multi_label=True
)

probability_sum = sum(multi_result["scores"])

difference = abs(probability_sum - 1)

print(
    f"Absolute Difference = {difference:.4f}"
)